In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("hw8.ipynb")

# CPSC 330 - Applied Machine Learning

## Homework 8: Introduction to Computer vision and Time Series

**Due date: See [deliverable due dates](https://ubc-cs.github.io/cpsc330-2025W2/#deliverable-due-dates-tentative)**.

## Imports

In [2]:
from hashlib import sha1

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
## Instructions
rubric={points}

You will earn points for following these instructions and successfully submitting your work on Gradescope.  

### Group wotk instructions

**You may work with a partner on this homework and submit your assignment as a group.** Below are some instructions on working as a group.  
- The maximum group size is 2.
  
- Use group work as an opportunity to collaborate and learn new things from each other. 
- Be respectful to each other and make sure you understand all the concepts in the assignment well. 
- It's your responsibility to make sure that the assignment is submitted by one of the group members before the deadline. 
- You can find the instructions on how to do group submission on Gradescope [here](https://help.gradescope.com/article/m5qz2xsnjy-student-add-group-members).
- If you would like to use late tokens for the homework, all group members must have the necessary late tokens available. Please note that the late tokens will be counted for all members of the group.   


### General submission instructions

- Please **read carefully
[Use of Generative AI policy](https://ubc-cs.github.io/cpsc330-2025W2/syllabus#use-of-generative-ai-in-the-course)** before starting the homework assignment. 
- **Run all cells before submitting:** Go to `Kernel -> Restart Kernel and Clear All Outputs`, then select `Run -> Run All Cells`. This ensures your notebook runs cleanly from start to finish without errors.
  
- **Submit your files on Gradescope.**  
   - Upload only your `.ipynb` file **with outputs displayed** and any required output files.
     
   - Do **not** submit other files from your repository.  
   - If you need help, see the [Gradescope Student Guide](https://lthub.ubc.ca/guides/gradescope-student-guide/).  
- **Check that outputs render properly.**  
   - Make sure all plots and outputs appear in your submission.
     
   - If your `.ipynb` file is too large and doesn't render on Gradescope, also upload a PDF or HTML version so the TAs can view your work.  
- **Keep execution order clean.**  
   - Execution numbers must start at "1" and increase in order.
     
   - Notebooks without visible outputs may not be graded.  
   - Out-of-order or missing execution numbers may result in mark deductions.  
- **Follow course submission guidelines:** Review the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions) for detailed guidance on completing and submitting assignments. 
   
</div>

_Points:_ 2

<!-- END QUESTION -->

<br><br>

## Exercise 1: time series prediction

In this exercise we'll be looking at a [dataset of avocado prices](https://www.kaggle.com/neuromusic/avocado-prices). You should start by downloading the dataset and storing it under the `data` folder. We will be forcasting average avocado price for the next week. 

In [3]:
df = pd.read_csv("data/avocado.csv", parse_dates=["Date"], index_col=0)
df.head()

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,2015-12-27,1.33,64236.62,1036.74,54454.85,48.16,8696.87,8603.62,93.25,0.0,conventional,2015,Albany
1,2015-12-20,1.35,54876.98,674.28,44638.81,58.33,9505.56,9408.07,97.49,0.0,conventional,2015,Albany
2,2015-12-13,0.93,118220.22,794.70,109149.67,130.50,8145.35,8042.21,103.14,0.0,conventional,2015,Albany
3,2015-12-06,1.08,78992.15,1132.00,71976.41,72.58,5811.16,5677.40,133.76,0.0,conventional,2015,Albany
4,2015-11-29,1.28,51039.60,941.48,43838.39,75.78,6183.95,5986.26,197.69,0.0,conventional,2015,Albany


In [4]:
df.shape

(18249, 13)

In [5]:
df["Date"].min()

Timestamp('2015-01-04 00:00:00')

In [6]:
df["Date"].max()

Timestamp('2018-03-25 00:00:00')

It looks like the data ranges from the start of 2015 to March 2018 (~2 years ago), for a total of 3.25 years or so. Let's split the data so that we have a 6 months of test data.

In [7]:
split_date = '20170925'
df_train = df[df["Date"] <= split_date]
df_test  = df[df["Date"] >  split_date]

In [8]:
assert len(df_train) + len(df_test) == len(df)

<br><br>

<!-- BEGIN QUESTION -->

### 1.1 How many time series? 
rubric={points:4}

In the [Rain in Australia](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package) dataset from lecture demo, we had different measurements for each Location. 

We want you to consider this for the avocado prices dataset. For which categorical feature(s), if any, do we have separate measurements? Justify your answer by referencing the dataset.

<div class="alert alert-warning">

Solution_1.1
    
</div>

_Points:_ 4

We have separate time series for each combination of `region` and `type` (conventional vs. organic). Both are categorical features that partition the dataset into independent measurement streams.   

For example, Albany-conventional and Albany-organic each have their own weekly price trajectories. Running `df.groupby(["region", "type"]).size()` confirms that each group has roughly the same number of rows (weekly measurements over ~3.25 years), indicating these are the axes along which separate time series exist.

In [16]:
print("Unique regions:", df["region"].nunique())
print("Unique types:", df["type"].nunique())
print("\nRows per (region, type) group:")
print(df.groupby(["region", "type"]).size().describe())

Unique regions: 54
Unique types: 2

Rows per (region, type) group:
count    108.000000
mean     168.972222
std        0.288675
min      166.000000
25%      169.000000
50%      169.000000
75%      169.000000
max      169.000000
dtype: float64


In [18]:
df.groupby(["region", "type"]).size().head(10)

region               type        
Albany               conventional    169
                     organic         169
Atlanta              conventional    169
                     organic         169
BaltimoreWashington  conventional    169
                     organic         169
Boise                conventional    169
                     organic         169
Boston               conventional    169
                     organic         169
dtype: int64

In [19]:
print(df[["region","type"]].dtypes)
print("\nUnique types:", df["type"].unique())

region    object
type      object
dtype: object

Unique types: ['conventional' 'organic']


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.2 Equally spaced measurements? 
rubric={points:4}

In the Rain in Australia dataset, the measurements were generally equally spaced but with some exceptions. How about with this dataset? Justify your answer by referencing the dataset.

<div class="alert alert-warning">

Solution_1.2
    
</div>

_Points:_ 4

The measurements are equally spaced at weekly (7-day) intervals within each (region, type) group. We can verify this by sorting the data by group and date, then computing the difference between consecutive dates within each group. Nearly all differences are exactly 7 days. There are a small number of gaps larger than 7 days at the boundary between groups (where one group ends and the next begins), but within a given (region, type) time series, the spacing is consistently 7 days.

In [25]:
df_sort = df.sort_values(by=["region", "type", "Date"]).reset_index(drop=True)
sample = df_sort[(df_sort["region"]=="Albany") & (df_sort["type"]=="conventional")].copy()
sample = sample.sort_values("Date")
diffs = sample["Date"].diff().dropna()

In [26]:
print("Date differences for Albany/conventional:")
print(diffs.value_counts())

Date differences for Albany/conventional:
Date
7 days    168
Name: count, dtype: int64


In [27]:
df_sort_copy = df_sort.copy()
df_sort_copy["date_diff"] = df_sort_copy.groupby(["region","type"])["Date"].diff()
within_group = df_sort_copy["date_diff"].dropna()

In [28]:
print("All within-group date differences:")
print(within_group.value_counts())

All within-group date differences:
date_diff
7 days     18139
14 days        1
21 days        1
Name: count, dtype: int64


In [29]:
pct_7days = (within_group == pd.Timedelta("7 days")).mean()
print(f"Fraction of gaps that are exactly 7 days: {pct_7days:.4f}")

Fraction of gaps that are exactly 7 days: 0.9999


In [ ]:
...

In [ ]:
...

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.3 Interpreting regions 
rubric={points:4}

In the Rain in Australia dataset, each location was a different place in Australia. For this dataset, look at the names of the regions. Do you think the regions are also all distinct, or are there overlapping regions? Justify your answer by referencing the data.

<div class="alert alert-warning">

Solution_1.3
    
</div>

_Points:_ 4

There are overlapping regions in this dataset. Within the "region" data set, there consists cities (e.g., `LosAngeles`, `Chicago`, etc.) and larger regions (e.g., `TotalUS`, `West`, `SouthCentral`, etc.), and because these cities are part of those larger regions (e.g., `Boston` is a part of `NorthernNewEngland`), there exists overlapping regions.

In [30]:
print(sorted(df["region"].unique()))

['Albany', 'Atlanta', 'BaltimoreWashington', 'Boise', 'Boston', 'BuffaloRochester', 'California', 'Charlotte', 'Chicago', 'CincinnatiDayton', 'Columbus', 'DallasFtWorth', 'Denver', 'Detroit', 'GrandRapids', 'GreatLakes', 'HarrisburgScranton', 'HartfordSpringfield', 'Houston', 'Indianapolis', 'Jacksonville', 'LasVegas', 'LosAngeles', 'Louisville', 'MiamiFtLauderdale', 'Midsouth', 'Nashville', 'NewOrleansMobile', 'NewYork', 'Northeast', 'NorthernNewEngland', 'Orlando', 'Philadelphia', 'PhoenixTucson', 'Pittsburgh', 'Plains', 'Portland', 'RaleighGreensboro', 'RichmondNorfolk', 'Roanoke', 'Sacramento', 'SanDiego', 'SanFrancisco', 'Seattle', 'SouthCarolina', 'SouthCentral', 'Southeast', 'Spokane', 'StLouis', 'Syracuse', 'Tampa', 'TotalUS', 'West', 'WestTexNewMexico']


In [31]:
vol = df.groupby("region")["Total Volume"].mean().sort_values(ascending=False)
print(vol.head(15))

region
TotalUS             1.735130e+07
West                3.215323e+06
California          3.044324e+06
SouthCentral        2.991952e+06
Northeast           2.110299e+06
Southeast           1.820232e+06
GreatLakes          1.744505e+06
Midsouth            1.503992e+06
LosAngeles          1.502653e+06
Plains              9.206761e+05
NewYork             7.122311e+05
DallasFtWorth       6.166251e+05
Houston             6.010884e+05
PhoenixTucson       5.788264e+05
WestTexNewMexico    4.314085e+05
Name: Total Volume, dtype: float64


In [32]:
aggregates = ["TotalUS","West","Northeast","Southeast","Plains","Midsouth",
              "SouthCentral","California","GreatLakes","WestTexNewMexico",
              "HartfordSpringfield","NorthernNewEngland"]
cities = [r for r in df["region"].unique() if r not in aggregates]
print(f"Aggregate-sounding regions ({len(aggregates)}): {aggregates}")
print(f"\nCity/metro regions ({len(cities)}): {cities}")

Aggregate-sounding regions (12): ['TotalUS', 'West', 'Northeast', 'Southeast', 'Plains', 'Midsouth', 'SouthCentral', 'California', 'GreatLakes', 'WestTexNewMexico', 'HartfordSpringfield', 'NorthernNewEngland']

City/metro regions (42): ['Albany', 'Atlanta', 'BaltimoreWashington', 'Boise', 'Boston', 'BuffaloRochester', 'Charlotte', 'Chicago', 'CincinnatiDayton', 'Columbus', 'DallasFtWorth', 'Denver', 'Detroit', 'GrandRapids', 'HarrisburgScranton', 'Houston', 'Indianapolis', 'Jacksonville', 'LasVegas', 'LosAngeles', 'Louisville', 'MiamiFtLauderdale', 'Nashville', 'NewOrleansMobile', 'NewYork', 'Orlando', 'Philadelphia', 'PhoenixTucson', 'Pittsburgh', 'Portland', 'RaleighGreensboro', 'RichmondNorfolk', 'Roanoke', 'Sacramento', 'SanDiego', 'SanFrancisco', 'Seattle', 'SouthCarolina', 'Spokane', 'StLouis', 'Syracuse', 'Tampa']


<!-- END QUESTION -->

<br><br>

We will use the entire dataset despite any location-based weirdness uncovered in the previous part.

We will be trying to forecast the avocado price. The function below is adapted from the lecture.

In [33]:
import pandas as pd


def create_lag_feature(
    df: pd.DataFrame,
    orig_feature: str,
    lag: int,
    groupby: list[str],
    new_feature_name: str | None = None,
    clip: bool = False,
) -> pd.DataFrame:
    """
    Create a lagged (or ahead) version of a feature, optionally per group.

    Assumes df is already sorted by time within each group and has unique indices.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset.
    orig_feature : str
        Name of the column to lag.
    lag : int
        The lag:
          - negative → values from the past (t-1, t-2, ...)
          - positive → values from the future (t+1, t+2, ...)
    groupby : list of str
        Column(s) to group by if df contains multiple time series.
    new_feature_name : str, optional
        Name of the new column. If None, a name is generated automatically.
    clip : bool, default False
        If True, drop rows where the new feature is NaN.

    Returns
    -------
    pd.DataFrame
        A new dataframe with the additional column added.
    """
    if lag == 0:
        raise ValueError("lag cannot be 0 (no shift). Use the original feature instead.")

    # Default name if not provided
    if new_feature_name is None:
        if lag < 0:
            new_feature_name = f"{orig_feature}_lag{abs(lag)}"
        else:
            new_feature_name = f"{orig_feature}_ahead{lag}"

    df = df.copy()

    # Map your convention (negative=past, positive=future) to pandas shift
    # pandas: shift(+k) → past, shift(-k) → future
    periods = abs(lag) if lag < 0 else -lag

    df[new_feature_name] = (
        df.groupby(groupby, sort=False)[orig_feature]
          .shift(periods)
    )

    if clip:
        df = df.dropna(subset=[new_feature_name])

    return df


We first sort our dataframe properly:

In [34]:
df_sort = df.sort_values(by=["region", "type", "Date"]).reset_index(drop=True)
df_sort

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,2015-01-04,1.22,40873.28,2819.50,28287.42,49.90,9716.46,9186.93,529.53,0.0,conventional,2015,Albany
1,2015-01-11,1.24,41195.08,1002.85,31640.34,127.12,8424.77,8036.04,388.73,0.0,conventional,2015,Albany
2,2015-01-18,1.17,44511.28,914.14,31540.32,135.77,11921.05,11651.09,269.96,0.0,conventional,2015,Albany
3,2015-01-25,1.06,45147.50,941.38,33196.16,164.14,10845.82,10103.35,742.47,0.0,conventional,2015,Albany
4,2015-02-01,0.99,70873.60,1353.90,60017.20,179.32,9323.18,9170.82,152.36,0.0,conventional,2015,Albany
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18244,2018-02-25,1.57,18421.24,1974.26,2482.65,0.00,13964.33,13698.27,266.06,0.0,organic,2018,WestTexNewMexico
18245,2018-03-04,1.54,17393.30,1832.24,1905.57,0.00,13655.49,13401.93,253.56,0.0,organic,2018,WestTexNewMexico
18246,2018-03-11,1.56,22128.42,2162.67,3194.25,8.93,16762.57,16510.32,252.25,0.0,organic,2018,WestTexNewMexico
18247,2018-03-18,1.56,15896.38,2055.35,1499.55,0.00,12341.48,12114.81,226.67,0.0,organic,2018,WestTexNewMexico


We then call `create_lag_feature`. This creates a new column in the dataset `AveragePriceNextWeek`, which is the following week's `AveragePrice`. We have set `clip=True` which means it will remove rows where the target would be missing.

In [35]:
df_hastarget = create_lag_feature(df_sort, "AveragePrice", +1, ["region", "type"], "AveragePriceNextWeek", clip=True)
df_hastarget

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region,AveragePriceNextWeek
0,2015-01-04,1.22,40873.28,2819.50,28287.42,49.90,9716.46,9186.93,529.53,0.0,conventional,2015,Albany,1.24
1,2015-01-11,1.24,41195.08,1002.85,31640.34,127.12,8424.77,8036.04,388.73,0.0,conventional,2015,Albany,1.17
2,2015-01-18,1.17,44511.28,914.14,31540.32,135.77,11921.05,11651.09,269.96,0.0,conventional,2015,Albany,1.06
3,2015-01-25,1.06,45147.50,941.38,33196.16,164.14,10845.82,10103.35,742.47,0.0,conventional,2015,Albany,0.99
4,2015-02-01,0.99,70873.60,1353.90,60017.20,179.32,9323.18,9170.82,152.36,0.0,conventional,2015,Albany,0.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18243,2018-02-18,1.56,17597.12,1892.05,1928.36,0.00,13776.71,13553.53,223.18,0.0,organic,2018,WestTexNewMexico,1.57
18244,2018-02-25,1.57,18421.24,1974.26,2482.65,0.00,13964.33,13698.27,266.06,0.0,organic,2018,WestTexNewMexico,1.54
18245,2018-03-04,1.54,17393.30,1832.24,1905.57,0.00,13655.49,13401.93,253.56,0.0,organic,2018,WestTexNewMexico,1.56
18246,2018-03-11,1.56,22128.42,2162.67,3194.25,8.93,16762.57,16510.32,252.25,0.0,organic,2018,WestTexNewMexico,1.56


Our goal is to predict `AveragePriceNextWeek`. 

Let's split the data:

In [36]:
df_train = df_hastarget[df_hastarget["Date"] <= split_date]
df_test  = df_hastarget[df_hastarget["Date"] >  split_date]

<br><br>

<!-- BEGIN QUESTION -->

### 1.4 `AveragePrice` baseline 
rubric={points}

Soon we will want to build some models to forecast the average avocado price a week in advance. Before we start with any ML though, let's try a baseline. Previously we used `DummyClassifier` or `DummyRegressor` as a baseline. This time, we'll do something else as a baseline: we'll assume the price stays the same from this week to next week. So, we'll set our prediction of "AveragePriceNextWeek" exactly equal to "AveragePrice", assuming no change. That is kind of like saying, "If it's raining today then I'm guessing it will be raining tomorrow". This simplistic approach will not get a great score but it's a good starting point for reference. If our model does worse that this, it must not be very good. 

Using this baseline approach, what $R^2$ do you get on the train and test data?

<div class="alert alert-warning">

Solution_1.4
    
</div>

_Points:_ 4

For the train data using the `AveragePriceNextWeek`, we get an $R^2$ of 0.829, and for the test data, we get $R^2$ of 0.763. This reveals a strong baseline that all models have to beat. 

In [37]:
train_r2 = r2_score(df_train["AveragePriceNextWeek"], df_train["AveragePrice"])
print(f"Baseline Train R²: {train_r2:.3f}")

Baseline Train R²: 0.829


In [38]:
test_r2 = r2_score(df_test["AveragePriceNextWeek"], df_test["AveragePrice"])
print(f"Baseline Test R²:  {test_r2:.3f}")

Baseline Test R²:  0.763


In [39]:
...

Ellipsis

In [40]:
...

Ellipsis

In [41]:
assert not train_r2 is None, "Are you using the correct variable name?"
assert not test_r2 is None, "Are you using the correct variable name?"
assert sha1(str(round(train_r2, 3)).encode('utf8')).hexdigest() == 'b1136fe2a8918904393ab6f40bfb3f38eac5fc39', "Your training score is not correct. Are you using the right features?"
assert sha1(str(round(test_r2, 3)).encode('utf8')).hexdigest() == 'cc24d9a9b567b491a56b42f7adc582f2eefa5907', "Your test score is not correct. Are you using the right features?"

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.5 Forecasting average avocado price
rubric={points:10}

Now that the baseline is done, let's build some models to forecast the average avocado price a week later. Experiment with a few approachs for encoding the date. Justify the decisions you make. Which approach worked best? Report your test score and briefly discuss your results.

Benchmark: you should be able to achieve $R^2$ of at least 0.79 on the test set. I got to 0.80, but not beyond that. Let me know if you do better!

Note: because we only have 2 splits here, we need to be a bit wary of overfitting on the test set. Try not to test on it a ridiculous number of times. If you are interested in some proper ways of dealing with this, see for example sklearn's [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html), which is like cross-validation for time series data.

<div class="alert alert-warning">

Solution_1.5
    
</div>

_Points:_ 10

Four different approaches were used, with two different models (`Ridge`, `RandomForest`) used for all approaches. Generally, `RandomForest` seemed to work better than `Ridge` because the data is non-linear (prices fluctuate dynamically), which fits better with a `RandomForest` model.

**Approach 1: `Ridge` w/ all features**  
This approach used `Ridge` using all features (lag, cyclical encoding). This approach had a train R² of 0.854 and a test R² of 0.800. There exists little overfitting and does beat the baseline model as well. 

**Approach 2: `RandomForest` w/ all features**  
This approach used the same as 1 but with `RandomForest` instead. Its train R² was 0.981 and test R² was 0.801. There exists to be overfitting present (test-train gap is bigger than approach 1), however, it does outperform both the baseline and `Ridge`, even with the overfitting present. 

**Approach 3: `Ridge` w/ only lag features**  
This approach used `Ridge` using only the lag features (no date encoding). Its train R² was 0.851 and test R² was 0.798. Although it performs slightly less than both approaches above, the difference is minimal, whilst also having the advantage of using way less features. 

From the different models that were used (`Ridge` w/ all features, `RandomForest` w/ all features, `Ridge` w/ lag features only), `RandomForest` using all features seemed to be the best model, as it achieved the best test data $R^2$ score of 0.801. However, when compared to its train $R^2$ score (0.981), it seems that the model is overfitting a lot more than approach 1 and 3. Also, all three scores seem to be very close to each other (around 0.8). Therefore, which approach to use may depend on different factors, like feature count, pure accuracy, or simplicity of the model itself.

In [58]:
# Feature engineering

# Lag features: create features to capture prices from 1, 2, and 4 wks ago
# Used to capture the dynamic nature of avocado prices
df_feat = create_lag_feature(df_hastarget, "AveragePrice", -1, ["region", "type"], clip=False)
df_feat = create_lag_feature(df_feat, "AveragePrice", -2, ["region", "type"], clip=False)
df_feat = create_lag_feature(df_feat, "AveragePrice", -4, ["region", "type"], clip=False)

# Cyclical month encoding: used to encode the months properly 
# (e.g., winter months are closer together (December-January))
df_feat["month"] = df_feat["Date"].dt.month
df_feat["year"] = df_feat["Date"].dt.year
df_feat["month_sin"] = np.sin(2 * np.pi * df_feat["month"] / 12)
df_feat["month_cos"] = np.cos(2 * np.pi * df_feat["month"] / 12)
df_feat["day_of_year"] = df_feat["Date"].dt.dayofyear
df_feat.head()

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,...,year,region,AveragePriceNextWeek,AveragePrice_lag1,AveragePrice_lag2,AveragePrice_lag4,month,month_sin,month_cos,day_of_year
0,2015-01-04,1.22,40873.28,2819.50,28287.42,49.90,9716.46,9186.93,529.53,0.0,...,2015,Albany,1.24,NaN,NaN,NaN,1,0.500000,0.866025,4
1,2015-01-11,1.24,41195.08,1002.85,31640.34,127.12,8424.77,8036.04,388.73,0.0,...,2015,Albany,1.17,1.22,NaN,NaN,1,0.500000,0.866025,11
2,2015-01-18,1.17,44511.28,914.14,31540.32,135.77,11921.05,11651.09,269.96,0.0,...,2015,Albany,1.06,1.24,1.22,NaN,1,0.500000,0.866025,18
3,2015-01-25,1.06,45147.50,941.38,33196.16,164.14,10845.82,10103.35,742.47,0.0,...,2015,Albany,0.99,1.17,1.24,NaN,1,0.500000,0.866025,25
4,2015-02-01,0.99,70873.60,1353.90,60017.20,179.32,9323.18,9170.82,152.36,0.0,...,2015,Albany,0.99,1.06,1.17,1.22,2,0.866025,0.500000,32


In [59]:
feature_cols = [
    "AveragePrice", "AveragePrice_lag1", "AveragePrice_lag2", "AveragePrice_lag4",
    "Total Volume", "4046", "4225", "4770",
    "Total Bags", "Small Bags", "Large Bags", "XLarge Bags",
    "year", "month_sin", "month_cos", "day_of_year"
]
target_col = "AveragePriceNextWeek"

df_train2 = df_feat[df_feat["Date"] <= split_date].dropna(subset=feature_cols)
df_test2  = df_feat[df_feat["Date"] >  split_date].dropna(subset=feature_cols)

X_train = df_train2[feature_cols]
y_train = df_train2[target_col]
X_test  = df_test2[feature_cols]
y_test  = df_test2[target_col]

print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")

Train: 15009 rows | Test: 2700 rows


In [60]:
# Approach 1: Ridge with all features
ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])
ridge_pipe.fit(X_train, y_train)
print(f"Ridge  - Train R²: {ridge_pipe.score(X_train, y_train):.3f}  Test R²: {ridge_pipe.score(X_test, y_test):.3f}")

Ridge  - Train R²: 0.854  Test R²: 0.800


In [61]:
# Approach 2: RandomForest with all features (best approach)
rf_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])
rf_pipe.fit(X_train, y_train)
rf_train_r2 = rf_pipe.score(X_train, y_train)
rf_test_r2  = rf_pipe.score(X_test, y_test)
print(f"RF     - Train R²: {rf_train_r2:.3f}  Test R²: {rf_test_r2:.3f}")

RF     - Train R²: 0.981  Test R²: 0.801


In [66]:
# Approach 4: Ridge - lag features only (no date encoding)
lag_only = ["AveragePrice", "AveragePrice_lag1", "AveragePrice_lag2", "AveragePrice_lag4"]
ridge_lag = Pipeline([("scaler", StandardScaler()), ("model", Ridge())])
ridge_lag.fit(X_train[lag_only], y_train)
print(f"Ridge (lag only) - Train R²: {ridge_lag.score(X_train[lag_only], y_train):.3f}  "
      f"Test R²: {ridge_lag.score(X_test[lag_only], y_test):.3f}")

Ridge (lag only) - Train R²: 0.851  Test R²: 0.798


In [67]:
print("="*55)
print(f"Baseline (no change)    Test R²: {test_r2:.3f}")
print(f"Ridge (lag only)        Test R²: {ridge_lag.score(X_test[lag_only], y_test):.3f}")
print(f"Ridge (full+cyclical)   Test R²: {ridge_pipe.score(X_test, y_test):.3f}")
print(f"RandomForest (full)     Test R²: {rf_test_r2:.3f}  ← best")
print("="*55)

Baseline (no change)    Test R²: 0.763
Ridge (lag only)        Test R²: 0.798
Ridge (full+cyclical)   Test R²: 0.800
RandomForest (full)     Test R²: 0.801  ← best


In [ ]:
...

In [ ]:
...

In [ ]:
...

In [ ]:
...

In [ ]:
...

In [ ]:
...

In [ ]:
...

In [ ]:
...

In [ ]:
...

In [ ]:
...

In [ ]:
...

In [ ]:
...

In [ ]:
...

In [ ]:
...

In [ ]:
...

In [ ]:
...

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 2: Short answer questions

<!-- BEGIN QUESTION -->

### 2.1 Time series

rubric={points:6}

The following questions pertain to Lecture 20 on time series data:

1. Sometimes a time series has missing time points or, worse, time points that are unequally spaced in general. Give an example of a real world situation where the time series data would have unequally spaced time points.
2. In class we discussed two approaches to using temporal information: encoding the date as one or more features, and creating lagged versions of features. Which of these (one/other/both/neither) two approaches would struggle with unequally spaced time points? Briefly justify your answer.
3. When studying time series modeling, we explored several ways to encode date information as a feature for the citibike dataset. When we used time of day as a numeric feature, the Ridge model was not able to capture the periodic pattern. Why? How did we tackle this problem? Briefly explain.

<div class="alert alert-warning">

Solution_2.1
    
</div>

_Points:_ 6

1. Any irregular visits and/or meetings would result in unequally spaced time points. One real-world example that pertains to feature engineering and data management would be things like doctor visits and overall health records. This is because health check-ups are irregular and sometimes infrequent, and patients are most likely to visit when they have the time, which can vary from day-to-day.
   
2. Lagged features would suffer the most from unequally spaced time points, as lagged features are based on a defined lag of the feature (e.g., one week ago, one month ago, etc.), which would most likely result in one lag having more data than another lag, potentially increasing bias in the dataset. Encoding the data as a feature (e.g., cyclical encoding) would not face this issue, as its focus is on simply rewriting the dates into something that makes more sense for things like the seasons; how equal the spacing is between dates does not affect the encoding of the dates.

3. When `time-of-day` (with values 0-23) is used on a linear relationship model, like `Ridge`, it separates times (e.g., 11pm and 12am (midnight)) that are otherwise close together because they are numerically apart. A fix to this is to convert the `time-of-day` data to have a cyclical (sine-cosine) encoding, which allows for a "circular" data set, having closer times together (e.g., hour 0 & 23 are now close), allowing `Ridge` to learn this circular pattern of `time-of-day`.

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.2 Computer vision 
rubric={points:6}

The following questions pertain to the lecture on multiclass classification and introduction to computer vision. 

1. How many parameters (coefficients and intercepts) will `sklearn`’s `LogisticRegression()` model learn for a four-class classification problem, assuming that you have 10 features? Briefly explain your answer.
2. In Lecture 19, we briefly discussed how neural networks are sort of like `sklearn`'s pipelines, in the sense that they involve multiple sequential transformations of the data, finally resulting in the prediction. Why was this property useful when it came to transfer learning?
3. Imagine that you have a small dataset with ~1000 images containing pictures and names of 50 different Computer Science faculty members from UBC. Your goal is to develop a reasonably accurate multi-class classification model for this task. Describe which model/technique you would use and briefly justify your choice in one to three sentences.

<div class="alert alert-warning">

Solution_2.2
    
</div>

_Points:_ 6

1. `LogisticRegression` uses a One-vs-Rest (OvR) strategy, meaning that it fits one binary classifier per class. Each binary classifier learns 10 coefficient (1 for each feature) plus an intercept. Therefore, with 4 binary classifiers, we would have: $4 \times 11 = 44 \space\text{parameters}$ in total.

2. Sequential transformations of the data, when it comes to transfer learning, allows for the learning to be reused from different layers. For example, the early layers learn general patterns in the data, and as the layers deepen, it learns more task-specific patterns that would pertain to the current task. For transfer learning, the earlier layers can be used to train a different task entirely, even with a much smaller dataset, resulting in saved time from training and computing data.

3. With ~1000 images of 50 diff. CS students, that would be around 20 images per class (or student). Due to the small image-per-class ratio, training any model from scratch would severely overfit the model. Therefore, using a pre-trained model (like transfer learning w/ a pre-trained CNN) would work best in this scenario. This is because using a pre-trained CNN means that the model would have access to encoded data about visual and facial features from m trained images; we simply replace the output layer with a 50-class softmax and fine-tune, achieving strong generalization even with very limited data.

<!-- END QUESTION -->

<br><br>

Before submitting your assignment, please make sure you have followed all the instructions in the Submission Instructions section at the top. 

Here is a quick checklist before submitting: 

- [ ] Restart kernel, clear outputs, and run all cells from top to bottom.  
- [ ] `.ipynb` file runs without errors and contains all outputs.  
- [ ] Only `.ipynb` and required output files are uploaded (no extra files).  
- [ ] Execution numbers start at **1** and are in order.  
- [ ] If `.ipynb` is too large and doesn't render on Gradescope, also upload a PDF/HTML version.  
- [ ] Reviewed the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions).  

![](img/eva-well-done.png)